# ETL Pipeline — Mini Data Warehouse ร้านพิซซ่า

**Extract → Clean → Transform → Integrate → Load → Validate**

Notebook นี้แปลงข้อมูลดิบ 3 แหล่งให้เป็น Star Schema พร้อมใช้งานบน BI Tool

| ขั้นตอน | สิ่งที่ทำ |
|---------|-----------|
| **1. Extract** | อ่านข้อมูลจาก CSV เชิงสัมพันธ์ และ Nested JSON จาก 2 REST API |
| **2. Clean** | ตรวจจับและแก้ปัญหาคุณภาพข้อมูล 7 ประเภท พร้อมบันทึกหลักฐานก่อน–หลัง |
| **3. Transform** | สร้าง Dimension 5 ตาราง และ Bridge Table 1 ตาราง |
| **4. Integrate** | เชื่อม 3 แหล่งเข้าด้วยกันเป็น Fact Table และคำนวณ Measures |
| **5. Load** | โหลดเข้า DuckDB |
| **6. Validate** | เทียบตัวเลขควบคุมก่อนและหลัง Load |

### หลักการที่ยึดตลอด Notebook

1. **ไม่แก้ไฟล์ต้นทาง** — `01_Raw_Data/` เป็น read-only ทุกการแก้ไขเกิดในหน่วยความจำแล้วเขียนออกที่ใหม่
2. **รันซ้ำได้ (Idempotent)** — สั่ง Run All กี่รอบก็ได้ผลเหมือนเดิม ไม่มีขั้นตอนที่ต้องแก้ด้วยมือ
3. **ทุกการแก้ไขต้องมีหลักฐาน** — ทุกการ Clean บันทึกจำนวนแถวก่อน–หลังลง Audit Log

> **ต้องรัน `02_ETL/fetch_sources.py` ก่อนอย่างน้อยหนึ่งครั้ง** เพื่อดึงข้อมูลจาก API มาเก็บไว้

---
## 0. ตั้งค่าเริ่มต้น

In [1]:
import json
import re
import sqlite3
import sys
import unicodedata
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

# หา root ของโปรเจกต์ให้ได้ไม่ว่าจะรัน notebook จากที่ไหน
CWD = Path.cwd()
ROOT = CWD if (CWD / "01_Raw_Data").exists() else CWD.parent
assert (ROOT / "01_Raw_Data").exists(), f"หา 01_Raw_Data ไม่เจอ (cwd={CWD})"

RAW      = ROOT / "01_Raw_Data"
# CSV ยอดขายอยู่ใต้ 01_Raw_Data ตามโครงสร้างที่โจทย์กำหนด
# แต่ยังรองรับ layout เดิมที่วางไว้ที่ root ด้วย เผื่อรันจากเครื่องที่ยังไม่ได้ย้าย
SALES    = RAW / "pizza_sales" if (RAW / "pizza_sales").exists() else ROOT / "pizza_sales"
DW       = ROOT / "03_Data_Warehouse"
DW_CSV   = DW / "csv"
DB_PATH  = DW / "pizza_dw.duckdb"

DW.mkdir(exist_ok=True)
DW_CSV.mkdir(exist_ok=True)

print("ROOT     :", ROOT)
print("SALES    :", SALES)
print("pandas   :", pd.__version__)
print("duckdb   :", duckdb.__version__)

ROOT     : C:\Users\Poobpub\Desktop\Data mine\Proj_Pizza\githubfromเพื่อน
SALES    : C:\Users\Poobpub\Desktop\Data mine\Proj_Pizza\githubfromเพื่อน\01_Raw_Data\pizza_sales
pandas   : 3.0.4
duckdb   : 1.5.5


### Audit Log — โครงสร้างสำหรับเก็บหลักฐานการแก้ไข

Requirement กำหนดให้แสดงว่า *"ปัญหาคุณภาพข้อมูลถูกตรวจพบและแก้ไขอย่างไร"*
จึงสร้างตัวเก็บบันทึกกลาง ทุกครั้งที่แก้ข้อมูลต้องเรียก `audit()` เพื่อบันทึกจำนวนก่อน–หลัง
ตอนจบจะสรุปออกมาเป็นตารางเดียวสำหรับใส่ในรายงาน

In [2]:
AUDIT: list[dict] = []

def audit(step: str, issue: str, detail: str, before, after, action: str) -> None:
    """บันทึกหลักฐานการแก้ปัญหาคุณภาพข้อมูล 1 รายการ"""
    AUDIT.append({
        "ขั้นตอน": step,
        "ประเภทปัญหา": issue,
        "รายละเอียด": detail,
        "ก่อน": before,
        "หลัง": after,
        "วิธีแก้": action,
    })
    print(f"  [{issue}] {detail}: {before} -> {after}")

def check(condition: bool, message: str) -> None:
    """ตรวจสอบเงื่อนไข ถ้าไม่ผ่านให้หยุด pipeline ทันที ไม่ปล่อยข้อมูลเสียเข้าคลัง"""
    if condition:
        print(f"  PASS  {message}")
    else:
        raise AssertionError(f"FAIL  {message}")

---
# 1. EXTRACT — นำเข้าข้อมูลจากแหล่งต้นทาง

ดึงข้อมูลจาก 3 แหล่งที่มีรูปแบบต่างกันสิ้นเชิง

## 1.1 แหล่ง #1 — ยอดขาย POS (CSV เชิงสัมพันธ์ 4 ตาราง)

**ปัญหาที่เจอทันทีตั้งแต่ยังไม่ทันอ่านไฟล์:** `pizza_types.csv` เปิดด้วย UTF-8 ไม่ได้
เพราะมี byte `0x91` ซึ่งเป็น smart quote ของ Windows-1252 อยู่ในชื่อวัตถุดิบ `'Nduja Salami`

เซลล์ถัดไปสาธิตให้เห็นว่ามันพังจริง แล้วค่อยแก้ด้วยการระบุ encoding ให้ถูก

In [3]:
# สาธิตปัญหา: อ่านแบบ default (UTF-8) จะ error
try:
    pd.read_csv(SALES / "pizza_types.csv")
    print("อ่านผ่าน — ไม่มีปัญหา encoding")
except UnicodeDecodeError as e:
    print(f"UnicodeDecodeError: {e}")
    raw_bytes = (SALES / "pizza_types.csv").read_bytes()
    pos = e.start
    print(f"\nbyte ที่มีปัญหา: {hex(raw_bytes[pos])} ที่ตำแหน่ง {pos}")
    print("บริบทรอบๆ:", raw_bytes[pos-38:pos+30])

UnicodeDecodeError: 'utf-8' codec can't decode byte 0x91 in position 1710: invalid start byte

byte ที่มีปัญหา: 0x91 ที่ตำแหน่ง 1710
บริบทรอบๆ: b'alabrese,The Calabrese Pizza,Supreme,"\x91Nduja Salami, Pancetta, Tomat'


In [4]:
# แก้: ระบุ cp1252 (Windows-1252) ซึ่งเป็น encoding จริงของไฟล์
orders_raw       = pd.read_csv(SALES / "orders.csv",        encoding="utf-8")
order_details_raw= pd.read_csv(SALES / "order_details.csv", encoding="utf-8")
pizzas_raw       = pd.read_csv(SALES / "pizzas.csv",        encoding="utf-8")
pizza_types_raw  = pd.read_csv(SALES / "pizza_types.csv",   encoding="cp1252")

audit("Extract", "Encoding ไม่ใช่ UTF-8",
      "pizza_types.csv มี byte 0x91 (smart quote แบบ cp1252)",
      "อ่านไม่ได้ (UnicodeDecodeError)", f"อ่านได้ {len(pizza_types_raw)} แถว",
      "ระบุ encoding='cp1252' ตอนอ่านไฟล์")

for name, df in [("orders", orders_raw), ("order_details", order_details_raw),
                 ("pizzas", pizzas_raw), ("pizza_types", pizza_types_raw)]:
    print(f"{name:<15} {df.shape[0]:>6,} แถว x {df.shape[1]} คอลัมน์")

  [Encoding ไม่ใช่ UTF-8] pizza_types.csv มี byte 0x91 (smart quote แบบ cp1252): อ่านไม่ได้ (UnicodeDecodeError) -> อ่านได้ 32 แถว
orders          21,350 แถว x 3 คอลัมน์
order_details   48,620 แถว x 4 คอลัมน์
pizzas              96 แถว x 4 คอลัมน์
pizza_types         32 แถว x 4 คอลัมน์


In [5]:
# ตรวจว่า smart quote ถูกอ่านมาถูกต้องจริงไหม
mask = pizza_types_raw["ingredients"].str.contains("Nduja", na=False)
print(pizza_types_raw.loc[mask, ["pizza_type_id", "name", "ingredients"]].to_string(index=False))

pizza_type_id                name                                                                ingredients
    calabrese The Calabrese Pizza ‘Nduja Salami, Pancetta, Tomatoes, Red Onions, Friggitello Peppers, Garlic


## 1.2 แหล่ง #2 — สภาพอากาศรายชั่วโมง (Nested JSON จาก Open-Meteo API)

**ปัญหาโครงสร้าง:** API ไม่ได้คืน array ของ record แต่คืน **array คู่ขนาน (columnar)**

```json
"hourly": {
  "time":           ["2015-01-01T00:00", "2015-01-01T01:00", ...],
  "temperature_2m": [-6.5, -6.6, ...]
}
```

ค่าลำดับที่ *i* ของทุก array คือข้อมูลของชั่วโมงเดียวกัน จึงต้อง **transpose**
ให้เป็นแถวก่อนถึงจะใช้งานได้ ถ้าโหลดเข้า DW ตรงๆ จะได้ตารางที่มีแถวเดียวและคอลัมน์เป็น list

In [6]:
weather_path = RAW / "weather" / "openmeteo_archive_chicago_2015.json"
with open(weather_path, encoding="utf-8") as f:
    weather_json = json.load(f)

print("โครงสร้างชั้นบนสุด:", list(weather_json.keys()))
print("\nmetadata ของสถานี:")
for k in ["latitude", "longitude", "elevation", "timezone",
          "timezone_abbreviation", "utc_offset_seconds"]:
    print(f"  {k:<22} = {weather_json[k]}")

print("\nhourly เป็น", type(weather_json["hourly"]).__name__,
      "ที่มี", len(weather_json["hourly"]), "keys")
print("แต่ละ key เป็น list ยาว", len(weather_json["hourly"]["time"]))

โครงสร้างชั้นบนสุด: ['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'hourly_units', 'hourly', 'daily_units', 'daily']

metadata ของสถานี:
  latitude               = 41.862915
  longitude              = -87.64877
  elevation              = 241.0
  timezone               = America/Chicago
  timezone_abbreviation  = GMT-5
  utc_offset_seconds     = -18000

hourly เป็น dict ที่มี 9 keys
แต่ละ key เป็น list ยาว 8760


In [7]:
# Transpose: dict-of-arrays -> DataFrame แบบ row-oriented
# pd.DataFrame รับ dict ที่ทุก value ยาวเท่ากันได้ตรงๆ จึงเป็นวิธีที่สั้นและปลอดภัยที่สุด
weather_hourly_raw = pd.DataFrame(weather_json["hourly"])
weather_daily_raw  = pd.DataFrame(weather_json["daily"])

audit("Extract", "โครงสร้าง Columnar ไม่ใช่ Row",
      "Open-Meteo คืน array คู่ขนาน โหลดเข้า DW ตรงๆ ไม่ได้",
      f"{len(weather_json['hourly'])} arrays คู่ขนาน",
      f"{len(weather_hourly_raw):,} แถว x {weather_hourly_raw.shape[1]} คอลัมน์",
      "Transpose ด้วย pd.DataFrame() ให้เป็น row-oriented")

print()
display(weather_hourly_raw.head(3))

  [โครงสร้าง Columnar ไม่ใช่ Row] Open-Meteo คืน array คู่ขนาน โหลดเข้า DW ตรงๆ ไม่ได้: 9 arrays คู่ขนาน -> 8,760 แถว x 9 คอลัมน์



,time,temperature_2m,apparent_temperature,relative_humidity_2m,precipitation,rain,snowfall,weather_code,wind_speed_10m
0,2015-01-01T00:00,-6.5,-14.5,46,0.0,0.0,0.0,0,30.1
1,2015-01-01T01:00,-6.6,-14.6,48,0.0,0.0,0.0,0,30.4
2,2015-01-01T02:00,-6.9,-15.0,50,0.0,0.0,0.0,0,30.8


In [8]:
# เก็บหน่วยที่ API ส่งมาไว้ใช้อ้างอิงตอนแปลงหน่วย
units = weather_json["hourly_units"]
print("หน่วยที่ API คืนมา (ระบบเมตริก):")
for k, v in units.items():
    print(f"  {k:<24} {v}")

หน่วยที่ API คืนมา (ระบบเมตริก):
  time                     iso8601
  temperature_2m           °C
  apparent_temperature     °C
  relative_humidity_2m     %
  precipitation            mm
  rain                     mm
  snowfall                 cm
  weather_code             wmo code
  wind_speed_10m           km/h


## 1.3 แหล่ง #3 — วันหยุดนักขัตฤกษ์ (Nested JSON จาก Nager.Date API)

ไฟล์นี้เป็น array ของ object ตรงไปตรงมากว่า แต่มีฟิลด์ซ้อน 2 ฟิลด์คือ
`counties` (list ของรหัสรัฐ หรือ `null`) และ `types` (list ของประเภทวันหยุด)

In [9]:
holiday_path = RAW / "holidays" / "nager_publicholidays_us_2015.json"
with open(holiday_path, encoding="utf-8") as f:
    holiday_json = json.load(f)

holidays_raw = pd.DataFrame(holiday_json)
print(f"{len(holidays_raw)} แถว | คอลัมน์: {list(holidays_raw.columns)}")
print()
display(holidays_raw)

16 แถว | คอลัมน์: ['date', 'localName', 'name', 'countryCode', 'fixed', 'global', 'counties', 'launchYear', 'types']



,date,localName,name,countryCode,fixed,global,counties,launchYear,types
0,2015-01-01,New Year's Day,New Year's Day,US,False,True,None,None,"[Public, Bank]"
1,2015-01-19,"Martin Luther King, Jr. Day","Martin Luther King, Jr. Day",US,False,True,None,None,"[Public, Bank]"
2,2015-02-12,Lincoln's Birthday,Lincoln's Birthday,US,False,False,"[US-CA, US-CT, US-IL, US-IN, US-KY, US-MI, US-...",None,[Observance]
3,2015-02-16,Washington's Birthday,Presidents Day,US,False,True,None,None,"[Public, Bank]"
4,2015-04-03,Good Friday,Good Friday,US,False,False,"[US-CT, US-DE, US-HI, US-IN, US-KY, US-LA, US-...",None,[Public]
5,2015-04-03,Good Friday,Good Friday,US,False,False,[US-TX],None,[Optional]
6,2015-05-08,Truman Day,Truman Day,US,False,False,[US-MO],None,"[School, Authorities]"
7,2015-05-25,Memorial Day,Memorial Day,US,False,True,None,None,"[Public, Bank]"
8,2015-07-03,Independence Day,Independence Day,US,False,True,None,None,"[Public, Bank]"
9,2015-09-07,Labor Day,Labour Day,US,False,True,None,None,"[Public, Bank]"


In [10]:
# โหลดตารางถอดรหัสสภาพอากาศ WMO
with open(RAW / "weather" / "wmo_weather_codes.json", encoding="utf-8") as f:
    wmo_codes = pd.DataFrame(json.load(f))

print(f"ตารางอ้างอิง WMO: {len(wmo_codes)} รหัส")
display(wmo_codes.head(6))

ตารางอ้างอิง WMO: 28 รหัส


,weather_code,description,condition_group
0,0,Clear sky,Clear
1,1,Mainly clear,Clear
2,2,Partly cloudy,Cloudy
3,3,Overcast,Cloudy
4,45,Fog,Fog
5,48,Depositing rime fog,Fog


---
# 2. CLEAN — ตรวจจับและแก้ไขปัญหาคุณภาพข้อมูล

ทุกหัวข้อในส่วนนี้จะ **แสดงหลักฐานว่าปัญหามีอยู่จริงก่อน** แล้วจึงแก้
และบันทึกผลลง Audit Log

## 2.1 ตรวจสอบสภาพข้อมูลเบื้องต้น (Data Profiling)

ก่อนแก้อะไรต้องรู้ก่อนว่าอะไรเสีย เซลล์นี้สแกนทุกตารางหา
ค่าว่าง แถวซ้ำ และคีย์ซ้ำ

In [11]:
def profile(df: pd.DataFrame, name: str, key: str | None = None) -> dict:
    # ข้อมูลจาก API มีคอลัมน์ที่เก็บ list (เช่น counties, types) ซึ่ง hash ไม่ได้
    # ต้องแปลงเป็น string ก่อน ไม่งั้น duplicated() จะ error
    hashable = df.copy()
    for col in hashable.columns:
        if hashable[col].map(lambda v: isinstance(v, (list, dict))).any():
            hashable[col] = hashable[col].astype(str)
    return {
        "ตาราง": name,
        "แถว": len(df),
        "คอลัมน์": df.shape[1],
        "ค่าว่างรวม": int(df.isna().sum().sum()),
        "แถวซ้ำทั้งแถว": int(hashable.duplicated().sum()),
        "คีย์ซ้ำ": int(df[key].duplicated().sum()) if key else None,
    }

profile_report = pd.DataFrame([
    profile(orders_raw,        "orders",         "order_id"),
    profile(order_details_raw, "order_details",  "order_details_id"),
    profile(pizzas_raw,        "pizzas",         "pizza_id"),
    profile(pizza_types_raw,   "pizza_types",    "pizza_type_id"),
    profile(weather_hourly_raw,"weather_hourly", "time"),
    profile(weather_daily_raw, "weather_daily",  "time"),
    profile(holidays_raw,      "holidays"),
])
display(profile_report)

,ตาราง,แถว,คอลัมน์,ค่าว่างรวม,แถวซ้ำทั้งแถว,คีย์ซ้ำ
0,orders,21350,3,0,0,0.0
1,order_details,48620,4,0,0,0.0
2,pizzas,96,4,0,0,0.0
3,pizza_types,32,4,0,0,0.0
4,weather_hourly,8760,9,0,0,0.0
5,weather_daily,365,14,0,0,0.0
6,holidays,16,9,26,0,NaN


**อ่านผลได้ว่า** ข้อมูลยอดขายและสภาพอากาศสะอาดในแง่ค่าว่างและแถวซ้ำ
แต่ปัญหาที่แท้จริงไม่ได้อยู่ในระดับแถวเดี่ยว — มันอยู่ที่ **ความสอดคล้องระหว่างตาราง**
และ **รูปแบบข้อมูล** ซึ่งการ profile แบบผิวเผินจับไม่ได้ หัวข้อถัดไปจึงเจาะเป็นเรื่องๆ

## 2.2 ปัญหา: วันหยุดซ้ำ ทำให้ Fact Table บานปลาย

ไฟล์วันหยุดมี 16 แถว แต่เมื่อนับวันที่ไม่ซ้ำกลับได้น้อยกว่า
เพราะ API แยกแถวตามรัฐที่ประกาศวันหยุด

**นี่คือปัญหาที่อันตรายที่สุดในโปรเจกต์นี้** เพราะถ้า join เข้า Fact Table โดยไม่แก้ก่อน
ยอดขายจะถูกนับซ้ำโดยที่ไม่มีอะไรฟ้อง error เลย

In [12]:
print(f"จำนวนแถว        : {len(holidays_raw)}")
print(f"จำนวนวันที่ไม่ซ้ำ : {holidays_raw['date'].nunique()}")
print()
dupes = holidays_raw[holidays_raw.duplicated("date", keep=False)]
display(dupes[["date", "localName", "global", "counties"]])

จำนวนแถว        : 16
จำนวนวันที่ไม่ซ้ำ : 13



,date,localName,global,counties
4,2015-04-03,Good Friday,False,"[US-CT, US-DE, US-HI, US-IN, US-KY, US-LA, US-..."
5,2015-04-03,Good Friday,False,[US-TX]
10,2015-10-12,Columbus Day,False,"[US-AL, US-AZ, US-CO, US-CT, US-GA, US-ID, US-..."
11,2015-10-12,Columbus Day,True,None
12,2015-10-12,Indigenous Peoples' Day,False,"[US-AK, US-HI, US-SD]"


In [13]:
# พิสูจน์ผลกระทบ: ลอง join แบบไม่ dedupe ดูว่าแถวบานแค่ไหน
naive = orders_raw.merge(holidays_raw[["date", "localName"]], on="date", how="left")
print(f"orders ก่อน join : {len(orders_raw):,} แถว")
print(f"orders หลัง join : {len(naive):,} แถว")
print(f"แถวที่งอกขึ้นมา   : {len(naive) - len(orders_raw):,}  <-- ยอดขายจะถูกนับซ้ำเท่านี้")

orders ก่อน join : 21,350 แถว
orders หลัง join : 21,414 แถว
แถวที่งอกขึ้นมา   : 64  <-- ยอดขายจะถูกนับซ้ำเท่านี้


### วิธีแก้ — ยุบให้เหลือ 1 แถวต่อ 1 วันที่

ต้องตัดสินใจว่าเมื่อวันเดียวมีหลายชื่อวันหยุด จะเก็บชื่อไหน
กฎที่ใช้คือ **ให้ความสำคัญกับวันหยุดระดับประเทศ (`global=True`) ก่อน**
เพราะมีผลต่อพฤติกรรมผู้บริโภคมากกว่าวันหยุดเฉพาะบางรัฐ
หากเสมอกันให้เรียงตามตัวอักษรเพื่อให้ผลลัพธ์คงที่ทุกครั้งที่รัน (deterministic)

In [14]:
holidays = (
    holidays_raw
    .assign(is_global=lambda d: d["global"].astype(bool))
    # global=True มาก่อน แล้วเรียงชื่อเพื่อให้ผลคงที่ทุกครั้งที่รัน
    .sort_values(["date", "is_global", "localName"], ascending=[True, False, True])
    .drop_duplicates("date", keep="first")
    .loc[:, ["date", "localName", "is_global", "types"]]
    .rename(columns={"localName": "holiday_name"})
    .reset_index(drop=True)
)

audit("Clean", "ข้อมูลซ้ำ (Duplicate Records)",
      "วันหยุดซ้ำวันที่เดียวกันจากการแยกตามรัฐ",
      f"{len(holidays_raw)} แถว", f"{len(holidays)} แถว",
      "ยุบเหลือ 1 แถวต่อวัน โดยให้ global=True มาก่อน")

check(holidays["date"].is_unique, "วันที่ในตารางวันหยุดไม่ซ้ำแล้ว")
display(holidays)

  [ข้อมูลซ้ำ (Duplicate Records)] วันหยุดซ้ำวันที่เดียวกันจากการแยกตามรัฐ: 16 แถว -> 13 แถว
  PASS  วันที่ในตารางวันหยุดไม่ซ้ำแล้ว


,date,holiday_name,is_global,types
0,2015-01-01,New Year's Day,True,"[Public, Bank]"
1,2015-01-19,"Martin Luther King, Jr. Day",True,"[Public, Bank]"
2,2015-02-12,Lincoln's Birthday,False,[Observance]
3,2015-02-16,Washington's Birthday,True,"[Public, Bank]"
4,2015-04-03,Good Friday,False,[Public]
5,2015-05-08,Truman Day,False,"[School, Authorities]"
6,2015-05-25,Memorial Day,True,"[Public, Bank]"
7,2015-07-03,Independence Day,True,"[Public, Bank]"
8,2015-09-07,Labor Day,True,"[Public, Bank]"
9,2015-10-12,Columbus Day,True,[Bank]


> **ข้อสังเกตสำหรับรายงาน** — วันชาติสหรัฐฯ ปรากฏเป็นวันที่ **3 ก.ค. 2015** ไม่ใช่ 4 ก.ค.
> เพราะปีนั้นวันที่ 4 ตรงกับวันเสาร์ ทางการจึงเลื่อนวันหยุดมาเป็นวันศุกร์
> ต้องอธิบายจุดนี้ตอนวิเคราะห์ ไม่งั้นจะดูเหมือนข้อมูลผิด

## 2.3 ปัญหา: หน่วยวัดไม่ตรงกับบริบทธุรกิจ

API คืนค่าเป็นระบบเมตริก แต่ธุรกิจเป็นร้านในสหรัฐฯ ที่ตั้งราคาเป็น USD
ผู้ใช้งาน Dashboard คิดเป็นองศาฟาเรนไฮต์และนิ้ว

ที่สับสนกว่านั้นคือ **ในไฟล์เดียวกันยังใช้หน่วยไม่ตรงกันเอง** —
ปริมาณฝนเป็นมิลลิเมตร แต่หิมะเป็นเซนติเมตร ถ้าเผลอเอามาบวกกันตรงๆ จะผิดทันที

In [15]:
CM_PER_INCH, MM_PER_INCH, KM_PER_MILE = 2.54, 25.4, 1.609344

def c_to_f(c):    return c * 9 / 5 + 32
def mm_to_in(mm): return mm / MM_PER_INCH
def cm_to_in(cm): return cm / CM_PER_INCH
def kmh_to_mph(k):return k / KM_PER_MILE

w = weather_hourly_raw.copy()
before_sample = w.loc[0, ["temperature_2m", "precipitation", "snowfall", "wind_speed_10m"]].to_dict()

w["temp_f"]          = c_to_f(w["temperature_2m"])
w["feels_like_f"]    = c_to_f(w["apparent_temperature"])
w["precip_in"]       = mm_to_in(w["precipitation"])
w["rain_in"]         = mm_to_in(w["rain"])
w["snow_in"]         = cm_to_in(w["snowfall"])          # <-- cm ไม่ใช่ mm
w["wind_mph"]        = kmh_to_mph(w["wind_speed_10m"])
w["humidity_pct"]    = w["relative_humidity_2m"]

audit("Clean", "หน่วยวัดไม่ตรงกัน",
      "อุณหภูมิ C, ฝน mm, หิมะ cm, ลม km/h",
      "หน่วยเมตริก (4 หน่วย)", "หน่วยอังกฤษ (F, inch, mph)",
      "แปลงหน่วยทั้งหมด โดยหิมะใช้ตัวหาร 2.54 แยกจากฝนที่ใช้ 25.4")

print("\nตรวจผลการแปลงจากแถวแรก:")
print(f"  {before_sample['temperature_2m']:>7.1f} C     -> {w.loc[0,'temp_f']:>7.2f} F")
print(f"  {before_sample['precipitation']:>7.1f} mm    -> {w.loc[0,'precip_in']:>7.4f} in")
print(f"  {before_sample['snowfall']:>7.1f} cm    -> {w.loc[0,'snow_in']:>7.4f} in")
print(f"  {before_sample['wind_speed_10m']:>7.1f} km/h  -> {w.loc[0,'wind_mph']:>7.2f} mph")

  [หน่วยวัดไม่ตรงกัน] อุณหภูมิ C, ฝน mm, หิมะ cm, ลม km/h: หน่วยเมตริก (4 หน่วย) -> หน่วยอังกฤษ (F, inch, mph)

ตรวจผลการแปลงจากแถวแรก:
     -6.5 C     ->   20.30 F
      0.0 mm    ->  0.0000 in
      0.0 cm    ->  0.0000 in
     30.1 km/h  ->   18.70 mph


In [16]:
# ตรวจความสมเหตุสมผลของค่าหลังแปลง (Range Check)
check(w["temp_f"].between(-40, 120).all(), "อุณหภูมิอยู่ในช่วงที่เป็นไปได้ของชิคาโก (-40 ถึง 120 F)")
check((w["precip_in"] >= 0).all(),          "ปริมาณฝนไม่ติดลบ")
check((w["snow_in"]   >= 0).all(),          "ปริมาณหิมะไม่ติดลบ")
check(w["humidity_pct"].between(0, 100).all(), "ความชื้นอยู่ระหว่าง 0-100%")

print(f"\nช่วงอุณหภูมิทั้งปี: {w['temp_f'].min():.1f} F ถึง {w['temp_f'].max():.1f} F")

  PASS  อุณหภูมิอยู่ในช่วงที่เป็นไปได้ของชิคาโก (-40 ถึง 120 F)
  PASS  ปริมาณฝนไม่ติดลบ
  PASS  ปริมาณหิมะไม่ติดลบ
  PASS  ความชื้นอยู่ระหว่าง 0-100%

ช่วงอุณหภูมิทั้งปี: -9.0 F ถึง 85.1 F


## 2.4 ปัญหา: รูปแบบวันที่ไม่ตรงกัน 3 แบบจาก 3 แหล่ง

| แหล่ง | รูปแบบ | ตัวอย่าง |
|-------|--------|----------|
| orders.csv | แยก 2 คอลัมน์ | `date='2015-01-01'`, `time='11:38:36'` |
| Open-Meteo | ISO 8601 มี `T` คั่น | `'2015-01-01T00:00'` |
| Nager.Date | เฉพาะวันที่ | `'2015-01-01'` |

ต้องทำให้เป็นชนิด datetime เดียวกันก่อนถึงจะ join ได้

In [17]:
print("ก่อนแปลง — ชนิดข้อมูลล้วนเป็น string:")
print(f"  orders.date      {orders_raw['date'].dtype}   ตัวอย่าง {orders_raw['date'].iloc[0]!r}")
print(f"  orders.time      {orders_raw['time'].dtype}   ตัวอย่าง {orders_raw['time'].iloc[0]!r}")
print(f"  weather.time     {w['time'].dtype}   ตัวอย่าง {w['time'].iloc[0]!r}")
print(f"  holidays.date    {holidays['date'].dtype}   ตัวอย่าง {holidays['date'].iloc[0]!r}")

ก่อนแปลง — ชนิดข้อมูลล้วนเป็น string:
  orders.date      str   ตัวอย่าง '2015-01-01'
  orders.time      str   ตัวอย่าง '11:38:36'
  weather.time     str   ตัวอย่าง '2015-01-01T00:00'
  holidays.date    str   ตัวอย่าง '2015-01-01'


In [18]:
# ระบุ format ชัดเจนทุกจุด ไม่ปล่อยให้ pandas เดาเอง
# การเดาอาจตีความ 01/02/2015 สลับวัน-เดือนได้ และทำให้ผลไม่คงที่ระหว่างเวอร์ชัน
orders = orders_raw.copy()
orders["order_ts"]   = pd.to_datetime(orders["date"] + " " + orders["time"],
                                      format="%Y-%m-%d %H:%M:%S")
orders["order_date"] = pd.to_datetime(orders["date"], format="%Y-%m-%d")
orders["order_hour"] = orders["order_ts"].dt.hour

w["weather_ts"]   = pd.to_datetime(w["time"], format="%Y-%m-%dT%H:%M")
w["weather_date"] = w["weather_ts"].dt.normalize()
w["weather_hour"] = w["weather_ts"].dt.hour

holidays["holiday_date"] = pd.to_datetime(holidays["date"], format="%Y-%m-%d")

audit("Clean", "รูปแบบวันที่ไม่ตรงกัน",
      "3 แหล่งใช้รูปแบบวันที่ต่างกัน 3 แบบ",
      "string 3 รูปแบบ", "datetime64 ทั้งหมด",
      "ระบุ format string ชัดเจนใน pd.to_datetime() ทุกจุด")

check(orders["order_ts"].notna().all(),  "แปลง timestamp ของ orders ได้ครบทุกแถว")
check(w["weather_ts"].notna().all(),     "แปลง timestamp ของ weather ได้ครบทุกแถว")
check(holidays["holiday_date"].notna().all(), "แปลงวันที่ของ holidays ได้ครบทุกแถว")

  [รูปแบบวันที่ไม่ตรงกัน] 3 แหล่งใช้รูปแบบวันที่ต่างกัน 3 แบบ: string 3 รูปแบบ -> datetime64 ทั้งหมด
  PASS  แปลง timestamp ของ orders ได้ครบทุกแถว
  PASS  แปลง timestamp ของ weather ได้ครบทุกแถว
  PASS  แปลงวันที่ของ holidays ได้ครบทุกแถว


## 2.5 ปัญหา: ค่าที่เป็นรหัสตัวเลข อ่านความหมายไม่ได้

`weather_code` เป็นรหัส WMO 4677 — เลข `0` คือฟ้าใส `73` คือหิมะตกปานกลาง
`95` คือพายุฝนฟ้าคะนอง ถ้าเอาเลขดิบไปแสดงบน Dashboard ผู้ใช้จะอ่านไม่รู้เรื่อง
ต้อง join กับตารางอ้างอิงเพื่อถอดความหมาย

In [19]:
print("ค่า weather_code ที่พบจริงในข้อมูล:", sorted(w["weather_code"].unique()))

before_null = w["weather_code"].isna().sum()
w = w.merge(wmo_codes, on="weather_code", how="left")

# ต้องถอดรหัสได้ครบทุกแถว ถ้าเหลือ null แปลว่าตารางอ้างอิงไม่ครบ
unmapped = w["condition_group"].isna().sum()
audit("Clean", "ค่าที่เป็นรหัส (Coded Values)",
      "weather_code เป็นตัวเลข WMO ที่ตีความเองไม่ได้",
      f"{w['weather_code'].nunique()} รหัสตัวเลข",
      f"{w['condition_group'].nunique()} กลุ่มสภาพอากาศที่อ่านได้",
      "join กับตารางอ้างอิง WMO 4677")

check(unmapped == 0, f"ถอดรหัสสภาพอากาศได้ครบทุกแถว (ถอดไม่ได้ {unmapped} แถว)")
display(w["condition_group"].value_counts().rename("จำนวนชั่วโมง").to_frame())

ค่า weather_code ที่พบจริงในข้อมูล: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(51), np.int64(53), np.int64(55), np.int64(61), np.int64(63), np.int64(65), np.int64(71), np.int64(73), np.int64(75)]
  [ค่าที่เป็นรหัส (Coded Values)] weather_code เป็นตัวเลข WMO ที่ตีความเองไม่ได้: 13 รหัสตัวเลข -> 4 กลุ่มสภาพอากาศที่อ่านได้
  PASS  ถอดรหัสสภาพอากาศได้ครบทุกแถว (ถอดไม่ได้ 0 แถว)


,จำนวนชั่วโมง
condition_group,
Cloudy,4166
Clear,3370
Rain,925
Snow,299


## 2.6 ปัญหา: Timezone ไม่ขยับตาม Daylight Saving Time

เราขอข้อมูลด้วย `timezone=America/Chicago` แต่ API ตอบกลับมาด้วย
`utc_offset_seconds` **ค่าเดียวคงที่ตลอดทั้งปี**

ชิคาโกจริงๆ ใช้ CST (UTC−6) ในฤดูหนาว และเปลี่ยนเป็น CDT (UTC−5) ช่วง DST
การที่ API ใช้ offset เดียวแปลว่า timestamp ในช่วงฤดูหนาว
**คลาดจากนาฬิกาท้องถิ่นจริงไป 1 ชั่วโมง**

In [20]:
offset_hours = weather_json["utc_offset_seconds"] / 3600
print(f"offset ที่ API ใช้ : UTC{offset_hours:+.0f}  ({weather_json['timezone_abbreviation']})")
print(f"จำนวนชั่วโมงที่ได้ : {len(weather_hourly_raw):,}")
print(f"365 วัน x 24 ชม.  : {365*24:,}")
print()

# ถ้ามี DST จริง จะต้องมีวันที่มี 23 ชม. และวันที่มี 25 ชม.
hours_per_day = w.groupby("weather_date").size()
print("การกระจายจำนวนชั่วโมงต่อวัน:", hours_per_day.value_counts().to_dict())
print("-> ทุกวันมี 24 ชม. เท่ากันหมด ยืนยันว่า API ไม่ได้จำลอง DST")

audit("Clean", "Timezone / DST",
      "API ใช้ UTC offset คงที่ ไม่ขยับตาม DST",
      f"UTC{offset_hours:+.0f} ตลอดปี",
      "บันทึกเป็นข้อจำกัด ไม่ปรับแก้ข้อมูล",
      "ยอมรับความคลาด 1 ชม. ช่วงฤดูหนาว และระบุไว้ในรายงาน")

offset ที่ API ใช้ : UTC-5  (GMT-5)
จำนวนชั่วโมงที่ได้ : 8,760
365 วัน x 24 ชม.  : 8,760

การกระจายจำนวนชั่วโมงต่อวัน: {24: 365}
-> ทุกวันมี 24 ชม. เท่ากันหมด ยืนยันว่า API ไม่ได้จำลอง DST
  [Timezone / DST] API ใช้ UTC offset คงที่ ไม่ขยับตาม DST: UTC-5 ตลอดปี -> บันทึกเป็นข้อจำกัด ไม่ปรับแก้ข้อมูล


> **ทำไมจึงเลือกไม่แก้** — การวิเคราะห์ของเราใช้ช่วงเวลากว้างระดับ *daypart*
> (มื้อกลางวัน / มื้อเย็น) ความคลาด 1 ชั่วโมงไม่ทำให้ข้อสรุปเปลี่ยน
> การดัดแปลงข้อมูลต้นทางกลับสร้างความเสี่ยงมากกว่าประโยชน์
> **การตัดสินใจไม่แก้ พร้อมเหตุผลและการบันทึกไว้ ถือเป็นการจัดการคุณภาพข้อมูลเช่นกัน**

## 2.7 ตรวจสอบความสอดคล้องของคีย์ (Referential Integrity)

ก่อนจะ join อะไรต้องพิสูจน์ก่อนว่าคีย์ตรงกันจริง
ถ้ามีคีย์กำพร้าแล้วเผลอใช้ inner join ข้อมูลจะหายเงียบๆ โดยไม่มีอะไรเตือน

In [21]:
fk_checks = [
    ("order_details.order_id -> orders",        set(order_details_raw["order_id"]) - set(orders_raw["order_id"])),
    ("order_details.pizza_id -> pizzas",        set(order_details_raw["pizza_id"]) - set(pizzas_raw["pizza_id"])),
    ("pizzas.pizza_type_id -> pizza_types",     set(pizzas_raw["pizza_type_id"]) - set(pizza_types_raw["pizza_type_id"])),
    ("orders ที่ไม่มีรายการสินค้า",              set(orders_raw["order_id"]) - set(order_details_raw["order_id"])),
]
for label, missing in fk_checks:
    check(len(missing) == 0, f"{label:<42} คีย์กำพร้า {len(missing)} รายการ")

  PASS  order_details.order_id -> orders           คีย์กำพร้า 0 รายการ
  PASS  order_details.pizza_id -> pizzas           คีย์กำพร้า 0 รายการ
  PASS  pizzas.pizza_type_id -> pizza_types        คีย์กำพร้า 0 รายการ
  PASS  orders ที่ไม่มีรายการสินค้า                คีย์กำพร้า 0 รายการ


In [22]:
# ตรวจว่าทุกออเดอร์หาสภาพอากาศ ณ ชั่วโมงนั้นเจอครบ
order_hours   = set(zip(orders["order_date"], orders["order_hour"]))
weather_hours = set(zip(w["weather_date"],   w["weather_hour"]))
missing_wx    = order_hours - weather_hours

check(len(missing_wx) == 0,
      f"ทุกชั่วโมงที่มีออเดอร์มีข้อมูลอากาศครบ (ขาด {len(missing_wx)} ชั่วโมง)")
print(f"\nชั่วโมงที่มีออเดอร์ : {len(order_hours):,} ชั่วโมงไม่ซ้ำ")
print(f"ชั่วโมงที่มีข้อมูลอากาศ: {len(weather_hours):,} ชั่วโมง")

  PASS  ทุกชั่วโมงที่มีออเดอร์มีข้อมูลอากาศครบ (ขาด 0 ชั่วโมง)

ชั่วโมงที่มีออเดอร์ : 4,181 ชั่วโมงไม่ซ้ำ
ชั่วโมงที่มีข้อมูลอากาศ: 8,760 ชั่วโมง


## 2.8 ตรวจค่าผิดปกติในตัวเลขที่จะกลายเป็น Measure

ตัวเลขที่จะนำไปคำนวณเงินต้องไม่มีค่าติดลบหรือศูนย์ที่ไม่สมเหตุสมผล

In [23]:
check((order_details_raw["quantity"] > 0).all(), "quantity เป็นบวกทุกแถว")
check((pizzas_raw["price"] > 0).all(),           "price เป็นบวกทุกแถว")
check(order_details_raw["quantity"].max() <= 10, "ไม่มี quantity ที่สูงผิดปกติ")

print(f"\nquantity: ค่าที่พบ = {sorted(order_details_raw['quantity'].unique())}")
print(f"price   : ${pizzas_raw['price'].min():.2f} - ${pizzas_raw['price'].max():.2f}")

  PASS  quantity เป็นบวกทุกแถว
  PASS  price เป็นบวกทุกแถว
  PASS  ไม่มี quantity ที่สูงผิดปกติ

quantity: ค่าที่พบ = [np.int64(1), np.int64(2), np.int64(3), np.int64(4)]
price   : $9.75 - $35.95


## 2.9 สรุปผลการทำความสะอาดข้อมูล

In [24]:
audit_df = pd.DataFrame(AUDIT)
display(audit_df)
print(f"\nแก้ไขปัญหาคุณภาพข้อมูลทั้งหมด {len(audit_df)} ประเภท")

,ขั้นตอน,ประเภทปัญหา,รายละเอียด,ก่อน,หลัง,วิธีแก้
0,Extract,Encoding ไม่ใช่ UTF-8,pizza_types.csv มี byte 0x91 (smart quote แบบ ...,อ่านไม่ได้ (UnicodeDecodeError),อ่านได้ 32 แถว,ระบุ encoding='cp1252' ตอนอ่านไฟล์
1,Extract,โครงสร้าง Columnar ไม่ใช่ Row,Open-Meteo คืน array คู่ขนาน โหลดเข้า DW ตรงๆ ...,9 arrays คู่ขนาน,"8,760 แถว x 9 คอลัมน์",Transpose ด้วย pd.DataFrame() ให้เป็น row-orie...
2,Clean,ข้อมูลซ้ำ (Duplicate Records),วันหยุดซ้ำวันที่เดียวกันจากการแยกตามรัฐ,16 แถว,13 แถว,ยุบเหลือ 1 แถวต่อวัน โดยให้ global=True มาก่อน
3,Clean,หน่วยวัดไม่ตรงกัน,"อุณหภูมิ C, ฝน mm, หิมะ cm, ลม km/h",หน่วยเมตริก (4 หน่วย),"หน่วยอังกฤษ (F, inch, mph)",แปลงหน่วยทั้งหมด โดยหิมะใช้ตัวหาร 2.54 แยกจากฝ...
4,Clean,รูปแบบวันที่ไม่ตรงกัน,3 แหล่งใช้รูปแบบวันที่ต่างกัน 3 แบบ,string 3 รูปแบบ,datetime64 ทั้งหมด,ระบุ format string ชัดเจนใน pd.to_datetime() ท...
5,Clean,ค่าที่เป็นรหัส (Coded Values),weather_code เป็นตัวเลข WMO ที่ตีความเองไม่ได้,13 รหัสตัวเลข,4 กลุ่มสภาพอากาศที่อ่านได้,join กับตารางอ้างอิง WMO 4677
6,Clean,Timezone / DST,API ใช้ UTC offset คงที่ ไม่ขยับตาม DST,UTC-5 ตลอดปี,บันทึกเป็นข้อจำกัด ไม่ปรับแก้ข้อมูล,ยอมรับความคลาด 1 ชม. ช่วงฤดูหนาว และระบุไว้ในร...



แก้ไขปัญหาคุณภาพข้อมูลทั้งหมด 7 ประเภท


---
# 3. TRANSFORM — สร้าง Dimension Tables

ออกแบบตาม **Star Schema** คือทำให้แต่ละ Dimension แบนราบ (denormalized)
ไม่แตกเป็นตารางย่อยซ้อนกันแบบ Snowflake
เพราะ Star Schema ทำให้ผู้ใช้ BI Tool ลากฟิลด์ใช้งานได้ทันทีโดยไม่ต้อง join หลายชั้น

| Dimension | จำนวนแถว | อธิบาย |
|-----------|----------|--------|
| `dim_date` | 365 | ทุกวันของปี 2015 รวมวันที่ร้านปิด พร้อมธงวันหยุด |
| `dim_time` | 24 | ชั่วโมงของวัน พร้อมการจัดกลุ่มมื้ออาหาร |
| `dim_pizza` | 96 | เมนู x ขนาด แบนรวมหมวดหมู่และชื่อเมนูไว้ในตารางเดียว |
| `dim_weather` | 23 | Mini-dimension จากการจัดกลุ่มสภาพอากาศ |
| `dim_ingredient` | 65 | วัตถุดิบแต่ละชนิด |
| `bridge_pizza_ingredient` | 181 | ตารางเชื่อมความสัมพันธ์แบบหลายต่อหลาย |

## 3.1 `dim_date` — มิติเวลา

**สำคัญ:** ต้องสร้างจาก **ปฏิทินเต็มปี 365 วัน** ไม่ใช่จากวันที่ที่มีในตารางขาย
เพราะร้านปิดบางวัน ถ้าสร้างจากข้อมูลขายเท่านั้น วันที่ปิดจะหายไปจากมิติ
แล้ว Dashboard จะไม่สามารถแสดงได้ว่า "วันนั้นขายได้ศูนย์บาท" ซึ่งเป็นข้อมูลสำคัญ

In [25]:
date_range = pd.date_range("2015-01-01", "2015-12-31", freq="D")

dim_date = pd.DataFrame({"full_date": date_range})
dim_date["date_key"]     = dim_date["full_date"].dt.strftime("%Y%m%d").astype(int)
dim_date["year"]         = dim_date["full_date"].dt.year
dim_date["quarter"]      = dim_date["full_date"].dt.quarter
dim_date["quarter_name"] = "Q" + dim_date["quarter"].astype(str)
dim_date["month"]        = dim_date["full_date"].dt.month
dim_date["month_name"]   = dim_date["full_date"].dt.strftime("%B")
dim_date["month_short"]  = dim_date["full_date"].dt.strftime("%b")
dim_date["year_month"]   = dim_date["full_date"].dt.strftime("%Y-%m")
dim_date["day_of_month"] = dim_date["full_date"].dt.day
dim_date["day_of_week"]  = dim_date["full_date"].dt.dayofweek + 1      # 1=จันทร์
dim_date["day_name"]     = dim_date["full_date"].dt.strftime("%A")
dim_date["day_short"]    = dim_date["full_date"].dt.strftime("%a")
dim_date["week_of_year"] = dim_date["full_date"].dt.isocalendar().week.astype(int)
dim_date["is_weekend"]   = dim_date["day_of_week"].isin([6, 7])
dim_date["season"]       = dim_date["month"].map(
    {12:"Winter",1:"Winter",2:"Winter", 3:"Spring",4:"Spring",5:"Spring",
     6:"Summer",7:"Summer",8:"Summer", 9:"Fall",10:"Fall",11:"Fall"})

print(f"dim_date: {len(dim_date)} แถว")

dim_date: 365 แถว


In [26]:
# ผนวกข้อมูลวันหยุด (ที่ dedupe แล้วในข้อ 2.2)
dim_date = dim_date.merge(
    holidays[["holiday_date", "holiday_name"]],
    left_on="full_date", right_on="holiday_date", how="left"
).drop(columns="holiday_date")

dim_date["is_holiday"] = dim_date["holiday_name"].notna()
dim_date["holiday_name"] = dim_date["holiday_name"].fillna("")

# ตรวจว่าจำนวนแถวไม่บาน = การ dedupe ได้ผลจริง
check(len(dim_date) == 365, f"dim_date ยังมี 365 แถว ไม่บานปลายจากการ join วันหยุด")
check(dim_date["is_holiday"].sum() == len(holidays),
      f"ธงวันหยุดติดครบ {len(holidays)} วัน")

display(dim_date[dim_date["is_holiday"]][
    ["full_date", "day_name", "holiday_name"]].to_string(index=False))

  PASS  dim_date ยังมี 365 แถว ไม่บานปลายจากการ join วันหยุด
  PASS  ธงวันหยุดติดครบ 13 วัน


" full_date  day_name                holiday_name\n2015-01-01  Thursday              New Year's Day\n2015-01-19    Monday Martin Luther King, Jr. Day\n2015-02-12  Thursday          Lincoln's Birthday\n2015-02-16    Monday       Washington's Birthday\n2015-04-03    Friday                 Good Friday\n2015-05-08    Friday                  Truman Day\n2015-05-25    Monday                Memorial Day\n2015-07-03    Friday            Independence Day\n2015-09-07    Monday                   Labor Day\n2015-10-12    Monday                Columbus Day\n2015-11-11 Wednesday                Veterans Day\n2015-11-26  Thursday            Thanksgiving Day\n2015-12-25    Friday               Christmas Day"

In [27]:
# ทำเครื่องหมายวันที่ร้านเปิดขายจริง เพื่อใช้คำนวณ M6 (ยอดขายต่อวันทำการ)
trading_days = set(orders["order_date"])
dim_date["is_trading_day"] = dim_date["full_date"].isin(trading_days)

closed = dim_date[~dim_date["is_trading_day"]]
print(f"วันทำการ : {dim_date['is_trading_day'].sum()} วัน")
print(f"วันที่ปิด : {len(closed)} วัน\n")
display(closed[["full_date", "day_name", "holiday_name"]].to_string(index=False))

วันทำการ : 358 วัน


วันที่ปิด : 7 วัน



' full_date day_name  holiday_name\n2015-09-24 Thursday              \n2015-09-25   Friday              \n2015-10-05   Monday              \n2015-10-12   Monday  Columbus Day\n2015-10-19   Monday              \n2015-10-26   Monday              \n2015-12-25   Friday Christmas Day'

> **ข้อค้นพบ** — ร้านปิดทุก **วันจันทร์ตลอดเดือนตุลาคม** (5, 12, 19, 26)
> รูปแบบที่เป็นระบบขนาดนี้เกิดจากความบังเอิญไม่ได้
> มีความเป็นไปได้สองทางคือร้านมีนโยบายหยุดวันจันทร์ในเดือนนั้น
> หรือ **ข้อมูลของวันเหล่านั้นสูญหายตอนส่งออกจากระบบ POS**
>
> ไม่ว่าจะเป็นกรณีใด ต้องใช้ **M6 (ยอดขายต่อวันทำการ)** แทนยอดรวมรายเดือน
> เมื่อเปรียบเทียบระหว่างเดือน ไม่เช่นนั้นเดือนตุลาคมจะดูแย่กว่าความเป็นจริง

## 3.2 `dim_time` — มิติช่วงเวลาในวัน

แยกออกจาก `dim_date` ตามหลัก Kimball เพราะเวลาในวันมีรอบการวนซ้ำ (24 ชั่วโมง)
ที่เป็นอิสระจากปฏิทิน การรวมไว้ตารางเดียวจะทำให้ dimension บวมเป็น 8,760 แถวโดยไม่จำเป็น

In [28]:
def classify_daypart(h: int) -> str:
    if  9 <= h < 11: return "Morning"
    if 11 <= h < 14: return "Lunch"
    if 14 <= h < 17: return "Afternoon"
    if 17 <= h < 21: return "Dinner"
    return "Late Night"

dim_time = pd.DataFrame({"hour_24": range(24)})
dim_time["time_key"]    = dim_time["hour_24"]
dim_time["hour_label"]  = dim_time["hour_24"].map(lambda h: f"{h:02d}:00-{h:02d}:59")
dim_time["hour_ampm"]   = dim_time["hour_24"].map(
    lambda h: f"{(h-1)%12+1} {'AM' if h < 12 else 'PM'}")
dim_time["daypart"]     = dim_time["hour_24"].map(classify_daypart)
dim_time["is_open"]     = dim_time["hour_24"].between(9, 23)

display(dim_time)

,hour_24,time_key,hour_label,hour_ampm,daypart,is_open
0,0,0,00:00-00:59,12 AM,Late Night,False
1,1,1,01:00-01:59,1 AM,Late Night,False
2,2,2,02:00-02:59,2 AM,Late Night,False
3,3,3,03:00-03:59,3 AM,Late Night,False
4,4,4,04:00-04:59,4 AM,Late Night,False
5,5,5,05:00-05:59,5 AM,Late Night,False
6,6,6,06:00-06:59,6 AM,Late Night,False
7,7,7,07:00-07:59,7 AM,Late Night,False
8,8,8,08:00-08:59,8 AM,Late Night,False
9,9,9,09:00-09:59,9 AM,Morning,True


## 3.3 `dim_pizza` — มิติสินค้า (แบนราบตามหลัก Star Schema)

ข้อมูลต้นทางแยกเป็น `pizzas` (96 แถว) และ `pizza_types` (32 แถว)
ซึ่งเป็นโครงสร้างแบบ **Snowflake** ในการออกแบบ DW เราจะ **รวมสองตารางเป็นตารางเดียว**
เพื่อให้เป็น Star Schema แท้ ผู้ใช้ BI จะได้ลากใช้ `category` ได้ทันที
ไม่ต้อง join ผ่านตารางกลาง

In [29]:
SIZE_FULL = {"S": "Small", "M": "Medium", "L": "Large",
             "XL": "X-Large", "XXL": "XX-Large"}
SIZE_RANK = {"S": 1, "M": 2, "L": 3, "XL": 4, "XXL": 5}

dim_pizza = (
    pizzas_raw
    .merge(pizza_types_raw, on="pizza_type_id", how="left", validate="many_to_one")
    .rename(columns={"name": "pizza_name", "size": "size_code", "price": "unit_price"})
)
dim_pizza.insert(0, "pizza_key", range(1, len(dim_pizza) + 1))
dim_pizza["size_name"]  = dim_pizza["size_code"].map(SIZE_FULL)
dim_pizza["size_rank"]  = dim_pizza["size_code"].map(SIZE_RANK)
# ตัดคำว่า "The ... Pizza" ออกให้ชื่อสั้นลง เวลาแสดงบนแกนกราฟจะไม่ยาวเกิน
dim_pizza["pizza_short_name"] = (
    dim_pizza["pizza_name"].str.replace(r"^The\s+", "", regex=True)
                           .str.replace(r"\s+Pizza$", "", regex=True))

dim_pizza = dim_pizza[["pizza_key", "pizza_id", "pizza_type_id", "pizza_name",
                       "pizza_short_name", "category", "size_code", "size_name",
                       "size_rank", "unit_price", "ingredients"]]

check(len(dim_pizza) == 96,               "dim_pizza มี 96 แถว")
check(dim_pizza["pizza_id"].is_unique,    "pizza_id ไม่ซ้ำ")
check(dim_pizza["category"].notna().all(),"ทุกเมนูมีหมวดหมู่")
display(dim_pizza.head())

  PASS  dim_pizza มี 96 แถว
  PASS  pizza_id ไม่ซ้ำ
  PASS  ทุกเมนูมีหมวดหมู่


,pizza_key,pizza_id,pizza_type_id,pizza_name,pizza_short_name,category,size_code,size_name,size_rank,unit_price,ingredients
0,1,bbq_ckn_s,bbq_ckn,The Barbecue Chicken Pizza,Barbecue Chicken,Chicken,S,Small,1,12.75,"Barbecued Chicken, Red Peppers, Green Peppers,..."
1,2,bbq_ckn_m,bbq_ckn,The Barbecue Chicken Pizza,Barbecue Chicken,Chicken,M,Medium,2,16.75,"Barbecued Chicken, Red Peppers, Green Peppers,..."
2,3,bbq_ckn_l,bbq_ckn,The Barbecue Chicken Pizza,Barbecue Chicken,Chicken,L,Large,3,20.75,"Barbecued Chicken, Red Peppers, Green Peppers,..."
3,4,cali_ckn_s,cali_ckn,The California Chicken Pizza,California Chicken,Chicken,S,Small,1,12.75,"Chicken, Artichoke, Spinach, Garlic, Jalapeno ..."
4,5,cali_ckn_m,cali_ckn,The California Chicken Pizza,California Chicken,Chicken,M,Medium,2,16.75,"Chicken, Artichoke, Spinach, Garlic, Jalapeno ..."


In [30]:
# ตรวจข้อสังเกตสำคัญ: ขนาด XL/XXL มีขายกี่เมนู
size_coverage = (dim_pizza.groupby("size_code")["pizza_type_id"]
                 .nunique().rename("จำนวนเมนูที่มีขนาดนี้").to_frame())
size_coverage["สัดส่วนจากเมนูทั้งหมด"] = (
    size_coverage["จำนวนเมนูที่มีขนาดนี้"] / dim_pizza["pizza_type_id"].nunique() * 100
).round(1).astype(str) + "%"
display(size_coverage.loc[["S", "M", "L", "XL", "XXL"]])

print("\nเมนูที่มีขนาด XL/XXL:")
print(dim_pizza[dim_pizza["size_code"].isin(["XL", "XXL"])]
      [["pizza_name", "size_code", "unit_price"]].to_string(index=False))

,จำนวนเมนูที่มีขนาดนี้,สัดส่วนจากเมนูทั้งหมด
size_code,,
S,32,100.0%
M,31,96.9%
L,31,96.9%
XL,1,3.1%
XXL,1,3.1%



เมนูที่มีขนาด XL/XXL:
     pizza_name size_code  unit_price
The Greek Pizza        XL       25.50
The Greek Pizza       XXL       35.95


> **ข้อค้นพบที่เปลี่ยนข้อสรุปทางธุรกิจ** — ขนาด XL และ XXL มีขาย
> **เพียงเมนูเดียวจาก 32 เมนู** คือ The Greek Pizza
>
> ดังนั้นตัวเลขยอดขาย XXL ที่ต่ำมาก **ไม่ได้แปลว่าลูกค้าไม่ต้องการขนาดใหญ่**
> แต่แปลว่า **ร้านแทบไม่เคยเสนอขายขนาดนั้น** ข้อเสนอแนะที่ถูกต้องจึงไม่ใช่
> "ควรเลิกขายขนาดใหญ่" แต่เป็น *"ควรทดลองเปิดขายขนาด XL กับเมนูขายดี
> เพื่อพิสูจน์ว่ามีอุปสงค์จริงหรือไม่"*
>
> กรณีนี้ใช้เป็นตัวอย่างในหัวข้อ **การตรวจสอบความถูกต้องของข้อสรุปจาก AI** ได้โดยตรง

## 3.4 `dim_weather` — Mini-Dimension ของสภาพอากาศ

อุณหภูมิเป็นค่าต่อเนื่องที่มีค่าไม่ซ้ำกันเป็นพันค่า ถ้าใส่ลง Dimension ตรงๆ
จะได้ตารางที่ใหญ่พอๆ กับ Fact ซึ่งผิดหลักการ

วิธีมาตรฐานตามแนวทาง Kimball คือสร้าง **Mini-Dimension** โดย
**จัดค่าต่อเนื่องเป็นช่วง (banding)** แล้วเก็บเฉพาะชุดค่าที่เกิดขึ้นจริงไม่ซ้ำกัน
ทำให้ dimension เหลือขนาดเล็กและใช้กรองบน Dashboard ได้สะดวก

In [31]:
def temp_band(f: float) -> str:
    if f < 32:  return "1. Freezing (<32F)"
    if f < 50:  return "2. Cold (32-50F)"
    if f < 70:  return "3. Mild (50-70F)"
    if f < 85:  return "4. Warm (70-85F)"
    return "5. Hot (>85F)"

def precip_level(inches: float) -> str:
    if inches == 0:    return "None"
    if inches < 0.1:   return "Light"
    if inches < 0.3:   return "Moderate"
    return "Heavy"

w["temp_band"]     = w["temp_f"].map(temp_band)
w["precip_level"]  = w["precip_in"].map(precip_level)
w["is_precipitating"] = w["precip_in"] > 0

# เก็บเฉพาะชุดค่าที่เกิดขึ้นจริง ไม่สร้างทุก combination ที่เป็นไปได้
weather_attrs = ["condition_group", "temp_band", "precip_level", "is_precipitating"]
dim_weather = (w[weather_attrs].drop_duplicates()
               .sort_values(weather_attrs).reset_index(drop=True))
dim_weather.insert(0, "weather_key", range(1, len(dim_weather) + 1))

possible = (w["condition_group"].nunique() * w["temp_band"].nunique()
            * w["precip_level"].nunique() * 2)
print(f"combination ที่เป็นไปได้ทางทฤษฎี : {possible}")
print(f"combination ที่เกิดขึ้นจริง       : {len(dim_weather)}")
print(f"ประหยัดพื้นที่ไปได้              : {(1 - len(dim_weather)/possible)*100:.0f}%")
display(dim_weather.head(10))

combination ที่เป็นไปได้ทางทฤษฎี : 160
combination ที่เกิดขึ้นจริง       : 23
ประหยัดพื้นที่ไปได้              : 86%


,weather_key,condition_group,temp_band,precip_level,is_precipitating
0,1,Clear,1. Freezing (<32F),None,False
1,2,Clear,2. Cold (32-50F),None,False
2,3,Clear,3. Mild (50-70F),None,False
3,4,Clear,4. Warm (70-85F),None,False
4,5,Cloudy,1. Freezing (<32F),None,False
5,6,Cloudy,2. Cold (32-50F),None,False
6,7,Cloudy,3. Mild (50-70F),None,False
7,8,Cloudy,4. Warm (70-85F),None,False
8,9,Cloudy,5. Hot (>85F),None,False
9,10,Rain,1. Freezing (<32F),Light,True


In [32]:
# ผูก weather_key กลับเข้าตารางรายชั่วโมง เพื่อใช้เป็นสะพานไปหา Fact
weather_hourly = w.merge(dim_weather, on=weather_attrs, how="left",
                         validate="many_to_one")

check(weather_hourly["weather_key"].notna().all(), "ทุกชั่วโมงได้รับ weather_key")
check(len(weather_hourly) == 8760,                 "ยังมี 8,760 ชั่วโมงเท่าเดิม")

weather_lookup = weather_hourly[["weather_date", "weather_hour", "weather_key",
                                 "temp_f", "feels_like_f", "precip_in",
                                 "snow_in", "wind_mph", "humidity_pct"]]
print(f"\nตารางเชื่อมสภาพอากาศ: {len(weather_lookup):,} แถว")

  PASS  ทุกชั่วโมงได้รับ weather_key
  PASS  ยังมี 8,760 ชั่วโมงเท่าเดิม

ตารางเชื่อมสภาพอากาศ: 8,760 แถว


## 3.5 `dim_ingredient` + `bridge_pizza_ingredient` — ความสัมพันธ์แบบหลายต่อหลาย

คอลัมน์ `ingredients` เก็บวัตถุดิบหลายชนิดรวมไว้ในช่องเดียวคั่นด้วยจุลภาค
ซึ่งเป็นรูปแบบที่ผิดหลัก First Normal Form และนำไปวิเคราะห์ไม่ได้เลย

ความสัมพันธ์นี้เป็นแบบ **หลายต่อหลาย** (พิซซ่าหนึ่งชนิดมีหลายวัตถุดิบ
และวัตถุดิบหนึ่งชนิดใช้ในพิซซ่าหลายเมนู) จึงต้องใช้ **Bridge Table**
ซึ่งเป็นรูปแบบมาตรฐานของ Kimball สำหรับกรณีนี้

In [33]:
print("รูปแบบต้นทาง — วัตถุดิบทั้งหมดยัดอยู่ในช่องเดียว:")
print(pizza_types_raw.loc[0, "ingredients"])

รูปแบบต้นทาง — วัตถุดิบทั้งหมดยัดอยู่ในช่องเดียว:
Barbecued Chicken, Red Peppers, Green Peppers, Tomatoes, Red Onions, Barbecue Sauce


In [34]:
def normalize_ingredient(s: str) -> str:
    """ทำชื่อวัตถุดิบให้เป็นมาตรฐานเดียวกัน เพื่อไม่ให้ชื่อเดียวกันถูกนับเป็นคนละชนิด"""
    s = unicodedata.normalize("NFKC", s)         # จัดการอักขระ unicode แปลกๆ
    s = s.replace("\u2019", "'").replace("\u2018", "'")   # smart quote -> quote ปกติ
    s = re.sub(r"\s+", " ", s).strip()            # ยุบช่องว่างซ้ำ
    return s

# แตกคอลัมน์ที่คั่นด้วยจุลภาคออกเป็นหลายแถว
bridge_raw = (
    pizza_types_raw[["pizza_type_id", "ingredients"]]
    .assign(ingredient=lambda d: d["ingredients"].str.split(","))
    .explode("ingredient")
)
bridge_raw["ingredient"] = bridge_raw["ingredient"].map(normalize_ingredient)

audit("Transform", "ค่าหลายค่าในช่องเดียว (ผิด 1NF)",
      "คอลัมน์ ingredients เก็บวัตถุดิบหลายชนิดคั่นด้วยจุลภาค",
      f"{len(pizza_types_raw)} แถว (ช่องละหลายค่า)",
      f"{len(bridge_raw)} แถว (ช่องละหนึ่งค่า)",
      "แตกด้วย str.split() + explode() แล้วทำชื่อให้เป็นมาตรฐาน")

  [ค่าหลายค่าในช่องเดียว (ผิด 1NF)] คอลัมน์ ingredients เก็บวัตถุดิบหลายชนิดคั่นด้วยจุลภาค: 32 แถว (ช่องละหลายค่า) -> 181 แถว (ช่องละหนึ่งค่า)


In [35]:
# สร้าง dim_ingredient จากรายชื่อที่ไม่ซ้ำ
dim_ingredient = (pd.DataFrame({"ingredient_name": sorted(bridge_raw["ingredient"].unique())})
                  .reset_index(drop=True))
dim_ingredient.insert(0, "ingredient_key", range(1, len(dim_ingredient) + 1))

# จัดหมวดวัตถุดิบ เพื่อให้วิเคราะห์เชิงจัดซื้อได้ละเอียดขึ้น
def ingredient_category(n: str) -> str:
    n = n.lower()
    if "cheese" in n or n in {"mozzarella", "feta", "gouda", "fontina", "ricotta",
                              "provolone", "parmigiano reggiano", "brie carre",
                              "goat cheese", "blue cheese", "gorgonzola piccante cheese"}:
        return "Cheese"
    if any(k in n for k in ["chicken","beef","bacon","pepperoni","sausage","ham",
                            "salami","prosciutto","pancetta","capocollo","chorizo",
                            "duck","nduja","soppressata","calabrese","barbecued",
                            "coarse sicilian salami","'nduja salami","luganega sausage"]):
        return "Meat"
    if "sauce" in n or "pesto" in n:
        return "Sauce"
    if any(k in n for k in ["shrimp","anchovies","calamari","clams","crayfish","tuna"]):
        return "Seafood"
    return "Vegetable & Other"

dim_ingredient["ingredient_category"] = dim_ingredient["ingredient_name"].map(ingredient_category)

print(f"dim_ingredient: {len(dim_ingredient)} ชนิด")
display(dim_ingredient["ingredient_category"].value_counts().rename("จำนวน").to_frame())

dim_ingredient: 65 ชนิด


,จำนวน
ingredient_category,
Vegetable & Other,27
Meat,18
Cheese,14
Sauce,5
Seafood,1


In [36]:
bridge_pizza_ingredient = (
    bridge_raw.merge(dim_ingredient, left_on="ingredient",
                     right_on="ingredient_name", how="left", validate="many_to_one")
    [["pizza_type_id", "ingredient_key"]]
    .drop_duplicates()
    .sort_values(["pizza_type_id", "ingredient_key"])
    .reset_index(drop=True)
)

check(bridge_pizza_ingredient["ingredient_key"].notna().all(),
      "ทุกวัตถุดิบใน bridge หา ingredient_key เจอ")
check(bridge_pizza_ingredient["pizza_type_id"].nunique() == 32,
      "ครบทั้ง 32 เมนู")

print(f"bridge_pizza_ingredient: {len(bridge_pizza_ingredient)} ความสัมพันธ์")
print(f"เฉลี่ย {len(bridge_pizza_ingredient)/32:.1f} วัตถุดิบต่อเมนู")

  PASS  ทุกวัตถุดิบใน bridge หา ingredient_key เจอ
  PASS  ครบทั้ง 32 เมนู
bridge_pizza_ingredient: 181 ความสัมพันธ์
เฉลี่ย 5.7 วัตถุดิบต่อเมนู


> ⚠️ **คำเตือนเรื่องการใช้งาน Bridge Table**
>
> การ join Fact ผ่าน Bridge จะทำให้ยอดขายถูกนับซ้ำเท่ากับจำนวนวัตถุดิบในเมนูนั้น
> (เช่น พิซซ่าที่มี 6 วัตถุดิบ ยอดขายจะถูกนับ 6 ครั้ง)
>
> ตารางนี้จึงมีไว้ตอบคำถามประเภท *"วัตถุดิบใดถูกใช้ในเมนูมากที่สุด"* เท่านั้น
> **ห้ามนำมารวมยอดขายเด็ดขาด** ต้องระบุข้อจำกัดนี้ไว้ในเอกสารประกอบ Dashboard

---
# 4. INTEGRATE — เชื่อมทุกแหล่งเข้าด้วยกันเป็น Fact Table

## นิยาม Grain ของ Fact Table

> **หนึ่งแถวใน `fact_sales_line` = พิซซ่าหนึ่งชนิดในหนึ่งขนาด ที่ปรากฏในหนึ่งบิล**

พิซซ่าชนิดและขนาดเดียวกันที่สั่งหลายถาดในบิลเดียว จะถูกรวมเป็นแถวเดียว
โดยเก็บจำนวนไว้ในคอลัมน์ `quantity` ตามที่ระบบ POS ต้นทางบันทึกมา

**เหตุการณ์ที่ Fact นี้บันทึก:** การขายสินค้าหนึ่งรายการในใบเสร็จ
ณ เวลาที่ลูกค้าสั่ง — เป็น Fact แบบ **Transaction Fact Table**

### จำนวนแถวที่ต้องได้: **48,620 แถว** เท่ากับจำนวนแถวใน `order_details` พอดี
การ join ทุกครั้งต้องไม่ทำให้ตัวเลขนี้เปลี่ยน

In [37]:
row_count_before = len(order_details_raw)
print(f"เริ่มต้นจาก order_details: {row_count_before:,} แถว")

fact = order_details_raw.copy()

# --- เชื่อมที่ 1: ข้อมูลหัวบิล (วันที่และเวลา) ---
fact = fact.merge(orders[["order_id", "order_date", "order_hour"]],
                  on="order_id", how="inner", validate="many_to_one")
check(len(fact) == row_count_before, f"หลังเชื่อม orders ยังมี {row_count_before:,} แถว")

# --- เชื่อมที่ 2: มิติสินค้า (ได้ราคาต่อหน่วยมาด้วย) ---
fact = fact.merge(dim_pizza[["pizza_key", "pizza_id", "unit_price"]],
                  on="pizza_id", how="inner", validate="many_to_one")
check(len(fact) == row_count_before, f"หลังเชื่อม dim_pizza ยังมี {row_count_before:,} แถว")

เริ่มต้นจาก order_details: 48,620 แถว
  PASS  หลังเชื่อม orders ยังมี 48,620 แถว


  PASS  หลังเชื่อม dim_pizza ยังมี 48,620 แถว


In [38]:
# --- เชื่อมที่ 3: มิติสภาพอากาศ (เชื่อมด้วยวันที่ + ชั่วโมง) ---
# นี่คือจุดที่ข้อมูลจาก 2 แหล่งที่ไม่เกี่ยวข้องกันเลยมาบรรจบกัน
fact = fact.merge(
    weather_lookup[["weather_date", "weather_hour", "weather_key"]],
    left_on=["order_date", "order_hour"],
    right_on=["weather_date", "weather_hour"],
    how="left", validate="many_to_one",
).drop(columns=["weather_date", "weather_hour"])

check(len(fact) == row_count_before, f"หลังเชื่อม weather ยังมี {row_count_before:,} แถว")
check(fact["weather_key"].notna().all(), "ทุกแถวได้ weather_key ครบ ไม่มีค่าว่าง")

  PASS  หลังเชื่อม weather ยังมี 48,620 แถว
  PASS  ทุกแถวได้ weather_key ครบ ไม่มีค่าว่าง


In [39]:
# --- สร้าง Foreign Key ที่เหลือ ---
fact["date_key"] = fact["order_date"].dt.strftime("%Y%m%d").astype(int)
fact["time_key"] = fact["order_hour"]

# ตรวจว่า FK ทุกตัวชี้ไปยัง Dimension ที่มีอยู่จริง
check(set(fact["date_key"])   <= set(dim_date["date_key"]),    "date_key ทุกค่ามีใน dim_date")
check(set(fact["time_key"])   <= set(dim_time["time_key"]),    "time_key ทุกค่ามีใน dim_time")
check(set(fact["pizza_key"])  <= set(dim_pizza["pizza_key"]),  "pizza_key ทุกค่ามีใน dim_pizza")
check(set(fact["weather_key"])<= set(dim_weather["weather_key"]),"weather_key ทุกค่ามีใน dim_weather")

  PASS  date_key ทุกค่ามีใน dim_date
  PASS  time_key ทุกค่ามีใน dim_time
  PASS  pizza_key ทุกค่ามีใน dim_pizza
  PASS  weather_key ทุกค่ามีใน dim_weather


## 4.1 คำนวณ Measures

เก็บเฉพาะ **Base Measure** ที่บวกกันได้ทุกมิติ (fully additive) ลงใน Fact
ส่วน Derived Measure อย่างยอดขายเฉลี่ยต่อบิลจะให้ BI Tool คำนวณตอนแสดงผล

**เหตุผล:** ถ้าเก็บค่าเฉลี่ยลงในแต่ละแถว แล้วผู้ใช้ลากไปรวมทั้งเดือน
ระบบจะเอาค่าเฉลี่ยมาบวกกัน ซึ่งไม่มีความหมายทางคณิตศาสตร์

In [40]:
# M1 — ยอดขายรายบรรทัด
fact["line_revenue"] = (fact["quantity"] * fact["unit_price"]).round(2)

fact_sales_line = (
    fact[["order_details_id", "order_id", "date_key", "time_key",
          "pizza_key", "weather_key", "quantity", "unit_price", "line_revenue"]]
    .rename(columns={"order_details_id": "sales_key"})
    .sort_values("sales_key").reset_index(drop=True)
)

print("Base Measures ที่เก็บใน Fact:")
print(f"  quantity      — จำนวนถาด (M2)")
print(f"  unit_price    — ราคาต่อหน่วย ณ เวลาขาย")
print(f"  line_revenue  — ยอดขายรายบรรทัด (M1)")
print()
print("Degenerate Dimension:")
print(f"  order_id      — ใช้นับจำนวนบิล (M3) ด้วย COUNT(DISTINCT)")
print()
display(fact_sales_line.head())

Base Measures ที่เก็บใน Fact:
  quantity      — จำนวนถาด (M2)
  unit_price    — ราคาต่อหน่วย ณ เวลาขาย
  line_revenue  — ยอดขายรายบรรทัด (M1)

Degenerate Dimension:
  order_id      — ใช้นับจำนวนบิล (M3) ด้วย COUNT(DISTINCT)



,sales_key,order_id,date_key,time_key,pizza_key,weather_key,quantity,unit_price,line_revenue
0,1,1,20150101,11,26,1,1,13.25,13.25
1,2,2,20150101,11,23,1,1,16.00,16.00
2,3,2,20150101,11,72,1,1,18.50,18.50
3,4,2,20150101,11,51,1,1,20.75,20.75
4,5,2,20150101,11,86,1,1,16.00,16.00


In [41]:
# ตรวจความถูกต้องขั้นสุดท้ายของ Fact
check(len(fact_sales_line) == 48_620,                  "Fact มี 48,620 แถวตามที่ออกแบบไว้")
check(fact_sales_line["sales_key"].is_unique,          "sales_key ไม่ซ้ำ (เป็น PK ได้)")
check(fact_sales_line.isna().sum().sum() == 0,         "ไม่มีค่าว่างใน Fact เลย")
check((fact_sales_line["line_revenue"] > 0).all(),     "ยอดขายทุกบรรทัดเป็นบวก")
check(fact_sales_line["order_id"].nunique() == 21_350, "จำนวนบิลไม่ซ้ำ = 21,350")

  PASS  Fact มี 48,620 แถวตามที่ออกแบบไว้
  PASS  sales_key ไม่ซ้ำ (เป็น PK ได้)
  PASS  ไม่มีค่าว่างใน Fact เลย
  PASS  ยอดขายทุกบรรทัดเป็นบวก
  PASS  จำนวนบิลไม่ซ้ำ = 21,350


---
# 5. LOAD — โหลดเข้า Data Warehouse

ใช้ **DuckDB** เป็นคลังข้อมูล เหตุผลที่เลือก

- เป็นฐานข้อมูลแบบ **คอลัมน์** ออกแบบมาเพื่องานวิเคราะห์โดยเฉพาะ เร็วกว่า SQLite มากในงาน aggregate
- เก็บทั้งคลังในไฟล์เดียว พกพาและแนบส่งได้ ไม่ต้องติดตั้ง server
- ต่อกับ Power BI / Tableau / Python ได้โดยตรง

**เขียนแบบ Full Refresh** — ลบตารางเดิมทิ้งแล้วสร้างใหม่ทั้งหมด
เพื่อรับประกันว่ารันกี่ครั้งผลลัพธ์ก็เหมือนเดิม ไม่มีข้อมูลค้างจากรอบก่อน

In [42]:
TABLES = {
    "dim_date":                dim_date,
    "dim_time":                dim_time,
    "dim_pizza":               dim_pizza,
    "dim_weather":             dim_weather,
    "dim_ingredient":          dim_ingredient,
    "bridge_pizza_ingredient": bridge_pizza_ingredient,
    "fact_sales_line":         fact_sales_line,
}

if DB_PATH.exists():
    DB_PATH.unlink()          # full refresh — เริ่มจากไฟล์ว่างทุกครั้ง

con = duckdb.connect(str(DB_PATH))

for name, df in TABLES.items():
    con.register("_staging", df)
    con.execute(f"CREATE OR REPLACE TABLE {name} AS SELECT * FROM _staging")
    con.unregister("_staging")
    n = con.execute(f"SELECT COUNT(*) FROM {name}").fetchone()[0]
    print(f"  {name:<26} {n:>7,} แถว")

print(f"\nคลังข้อมูล -> {DB_PATH}")

  dim_date                       365 แถว
  dim_time                        24 แถว
  dim_pizza                       96 แถว
  dim_weather                     23 แถว
  dim_ingredient                  65 แถว
  bridge_pizza_ingredient        181 แถว
  fact_sales_line             48,620 แถว

คลังข้อมูล -> C:\Users\Poobpub\Desktop\Data mine\Proj_Pizza\githubfromเพื่อน\03_Data_Warehouse\pizza_dw.duckdb


In [43]:
# ส่งออกเป็น CSV ด้วย เผื่อ BI Tool บางตัวต่อ DuckDB ไม่ได้ (เช่น Looker Studio)
for name, df in TABLES.items():
    df.to_csv(DW_CSV / f"{name}.csv", index=False, encoding="utf-8-sig")
print(f"ส่งออก CSV {len(TABLES)} ไฟล์ -> {DW_CSV}")

ส่งออก CSV 7 ไฟล์ -> C:\Users\Poobpub\Desktop\Data mine\Proj_Pizza\githubfromเพื่อน\03_Data_Warehouse\csv


---
# 6. VALIDATE — ตรวจสอบความถูกต้องหลัง Load

Requirement กำหนดให้ *"มีการตรวจสอบความถูกต้องก่อนและหลัง Load"*

วิธีที่ใช้คือ **Control Totals** — คำนวณตัวเลขสำคัญจาก **ไฟล์ต้นทางโดยตรง**
แล้วเทียบกับผลที่ query ออกมาจากคลังข้อมูล ถ้าตัวเลขไม่ตรงกันแม้แต่ตัวเดียว
แปลว่ามีข้อมูลตกหล่นหรือถูกนับซ้ำระหว่างทาง

In [44]:
# ---- ฝั่งต้นทาง: คำนวณจาก CSV ดิบ ไม่ผ่าน pipeline เลย ----
src = (order_details_raw
       .merge(pizzas_raw, on="pizza_id")
       .merge(orders_raw, on="order_id"))
src_revenue  = (src["quantity"] * src["price"]).sum()
src_quantity = src["quantity"].sum()
src_orders   = orders_raw["order_id"].nunique()
src_rows     = len(order_details_raw)

# ---- ฝั่งปลายทาง: query จาก DuckDB ----
dw = con.execute("""
    SELECT ROUND(SUM(line_revenue), 2) AS revenue,
           SUM(quantity)               AS quantity,
           COUNT(DISTINCT order_id)    AS orders,
           COUNT(*)                    AS rows
    FROM fact_sales_line
""").fetchone()

comparison = pd.DataFrame([
    {"ตัวชี้วัด": "Total Revenue (M1)",  "ต้นทาง": round(src_revenue, 2), "ในคลังข้อมูล": dw[0]},
    {"ตัวชี้วัด": "Total Quantity (M2)", "ต้นทาง": int(src_quantity),     "ในคลังข้อมูล": dw[1]},
    {"ตัวชี้วัด": "Number of Orders (M3)","ต้นทาง": int(src_orders),      "ในคลังข้อมูล": dw[2]},
    {"ตัวชี้วัด": "จำนวนแถวใน Fact",      "ต้นทาง": int(src_rows),        "ในคลังข้อมูล": dw[3]},
])
comparison["ผลต่าง"] = comparison["ในคลังข้อมูล"] - comparison["ต้นทาง"]
comparison["ผล"]     = np.where(comparison["ผลต่าง"].abs() < 0.01, "PASS", "FAIL")
display(comparison)

check((comparison["ผล"] == "PASS").all(), "ตัวเลขควบคุมตรงกันทุกตัวระหว่างต้นทางกับคลังข้อมูล")

,ตัวชี้วัด,ต้นทาง,ในคลังข้อมูล,ผลต่าง,ผล
0,Total Revenue (M1),817860.05,817860.05,0.0,PASS
1,Total Quantity (M2),49574.00,49574.00,0.0,PASS
2,Number of Orders (M3),21350.00,21350.00,0.0,PASS
3,จำนวนแถวใน Fact,48620.00,48620.00,0.0,PASS


  PASS  ตัวเลขควบคุมตรงกันทุกตัวระหว่างต้นทางกับคลังข้อมูล


In [45]:
# ตรวจความสมบูรณ์เชิงความสัมพันธ์ในคลังข้อมูลจริง (Referential Integrity หลัง Load)
ri = con.execute("""
    SELECT
      (SELECT COUNT(*) FROM fact_sales_line f
        LEFT JOIN dim_date d    ON f.date_key    = d.date_key    WHERE d.date_key    IS NULL) AS orphan_date,
      (SELECT COUNT(*) FROM fact_sales_line f
        LEFT JOIN dim_time t    ON f.time_key    = t.time_key    WHERE t.time_key    IS NULL) AS orphan_time,
      (SELECT COUNT(*) FROM fact_sales_line f
        LEFT JOIN dim_pizza p   ON f.pizza_key   = p.pizza_key   WHERE p.pizza_key   IS NULL) AS orphan_pizza,
      (SELECT COUNT(*) FROM fact_sales_line f
        LEFT JOIN dim_weather wd ON f.weather_key = wd.weather_key WHERE wd.weather_key IS NULL) AS orphan_weather
""").fetchdf()
display(ri)
check(int(ri.sum(axis=1).iloc[0]) == 0, "ไม่มีแถวกำพร้าใน Fact — FK ทุกตัวชี้ไป Dimension ที่มีจริง")

,orphan_date,orphan_time,orphan_pizza,orphan_weather
0,0,0,0,0


  PASS  ไม่มีแถวกำพร้าใน Fact — FK ทุกตัวชี้ไป Dimension ที่มีจริง


## 6.1 ทดสอบตอบ Business Question ด้วย SQL จริง

พิสูจน์ว่าคลังข้อมูลที่สร้างเสร็จตอบคำถามทางธุรกิจได้จริง
ไม่ใช่แค่โหลดตารางเข้าไปเฉยๆ

### BQ5 — สภาพอากาศมีผลต่อยอดขายหรือไม่

คำถามนี้เป็นเหตุผลหลักที่เราดึงข้อมูลจาก Open-Meteo API เข้ามา
เพราะข้อมูล POS เพียงลำพังตอบไม่ได้เลย

**แต่คำถามนี้มีกับดักเรื่อง Grain ซ่อนอยู่** — สภาพอากาศเปลี่ยนได้ทุกชั่วโมง
วันหนึ่งจึงมีได้หลายสภาพอากาศ การเฉลี่ย "ต่อวัน" จึงผิด เพราะวันเดียวกัน
จะถูกนับซ้ำในหลายกลุ่ม เซลล์ถัดไปพิสูจน์ให้เห็นก่อนว่าผิดอย่างไร

In [46]:
# วิธีที่ผิด: หารด้วยจำนวนวัน
wrong = con.execute("""
    SELECT  wd.condition_group                     AS สภาพอากาศ,
            COUNT(DISTINCT f.date_key)             AS จำนวนวัน,
            ROUND(SUM(f.line_revenue)
                  / COUNT(DISTINCT f.date_key), 2) AS ยอดขายเฉลี่ยต่อวัน
    FROM fact_sales_line f
    JOIN dim_weather wd ON f.weather_key = wd.weather_key
    GROUP BY 1 ORDER BY 3 DESC
""").fetchdf()
display(wrong)

total_days_counted = wrong["จำนวนวัน"].sum()
actual_trading_days = int(dim_date["is_trading_day"].sum())
print(f"ผลรวมจำนวนวันจากทุกกลุ่ม : {total_days_counted}")
print(f"วันทำการจริงทั้งปี        : {actual_trading_days}")
print(f"-> ตัวหารซ้ำเกินความจริง {total_days_counted - actual_trading_days} วัน")
print("   ตัวเลข 'ต่อวัน' ข้างบนจึงเทียบกันไม่ได้")

,สภาพอากาศ,จำนวนวัน,ยอดขายเฉลี่ยต่อวัน
0,Cloudy,306,1344.74
1,Clear,233,1263.52
2,Rain,105,820.21
3,Snow,34,760.23


ผลรวมจำนวนวันจากทุกกลุ่ม : 678
วันทำการจริงทั้งปี        : 358
-> ตัวหารซ้ำเกินความจริง 320 วัน
   ตัวเลข 'ต่อวัน' ข้างบนจึงเทียบกันไม่ได้


**ทำไมถึงผิด** — วันฝนตกส่วนใหญ่ฝนตกแค่ไม่กี่ชั่วโมง เศษเงินที่หารมาจึงเป็น
รายได้เฉพาะชั่วโมงที่ฝนตก แต่ตัวหารกลับเป็น "ทั้งวัน" ทำให้สภาพอากาศที่เกิดสั้นๆ
(ฝน หิมะ) ดูเหมือนมียอดขายต่ำกว่าความจริงอย่างมาก

**วิธีที่ถูกคือเทียบต่อชั่วโมง** เพราะ `weather_key` ผูกกับ Fact ที่ระดับชั่วโมง
ตัวหารจึงต้องเป็นชั่วโมงด้วย จึงจะเทียบกันได้อย่างเป็นธรรม

In [47]:
# วิธีที่ถูก: หารด้วยจำนวนชั่วโมงที่เกิดสภาพอากาศนั้นจริง
display(con.execute("""
    SELECT  wd.condition_group                              AS สภาพอากาศ,
            COUNT(DISTINCT (f.date_key, f.time_key))        AS จำนวนชั่วโมง,
            ROUND(SUM(f.line_revenue), 2)                   AS ยอดขายรวม,
            ROUND(SUM(f.line_revenue)
                  / COUNT(DISTINCT (f.date_key, f.time_key)), 2) AS ยอดขายต่อชั่วโมง,
            ROUND(COUNT(DISTINCT f.order_id) * 1.0
                  / COUNT(DISTINCT (f.date_key, f.time_key)), 2) AS บิลต่อชั่วโมง,
            ROUND(SUM(f.line_revenue)
                  / COUNT(DISTINCT f.order_id), 2)          AS ยอดขายต่อบิล
    FROM fact_sales_line f
    JOIN dim_weather wd ON f.weather_key = wd.weather_key
    GROUP BY 1
    ORDER BY 4 DESC
""").fetchdf())

,สภาพอากาศ,จำนวนชั่วโมง,ยอดขายรวม,ยอดขายต่อชั่วโมง,บิลต่อชั่วโมง,ยอดขายต่อบิล
0,Rain,437,86121.95,197.08,5.16,38.21
1,Cloudy,2093,411491.05,196.60,5.11,38.47
2,Clear,1511,294399.40,194.84,5.10,38.17
3,Snow,140,25847.65,184.63,4.91,37.57


### ข้อสรุปของ BQ5 — สภาพอากาศแทบไม่มีผลต่อยอดขาย

เมื่อเทียบอย่างถูกวิธีแล้ว ยอดขายต่อชั่วโมงของทุกสภาพอากาศ
**ต่างกันไม่ถึง 7%** และฝนตกกลับให้ยอดต่อชั่วโมงสูงที่สุดเล็กน้อย
ส่วนยอดขายต่อบิลแทบไม่ขยับเลย (ราว $38 ทุกสภาพอากาศ)

**นี่คือผลลัพธ์เชิงลบที่มีคุณค่า** เพราะตอบคำถามผู้บริหารได้ว่า
*"ไม่ต้องเสียเวลาปรับแผนกำลังคนหรือสต็อกตามพยากรณ์อากาศ
เพราะพิสูจน์จากข้อมูลจริงทั้งปีแล้วว่าผลกระทบเล็กเกินกว่าจะคุ้มค่าบริหาร"*
ซึ่งมีประโยชน์กว่าการฝืนหาความสัมพันธ์ที่ไม่มีอยู่จริง

> **บทเรียนสำหรับรายงาน** — ถ้าดูแต่ตัวเลข "ต่อวัน" จะสรุปผิดว่า
> *"ฝนตกยอดหาย 39%"* ซึ่งเป็นภาพลวงจาก Grain ที่ไม่ตรงกันล้วนๆ
> ใช้เป็นตัวอย่างการตรวจสอบความถูกต้องของข้อสรุปได้อีกกรณีหนึ่ง

In [48]:
print("BQ3 — ช่วงเวลาใดหนาแน่นที่สุด\n")
display(con.execute("""
    SELECT  t.daypart                            AS ช่วงเวลา,
            COUNT(DISTINCT f.order_id)           AS จำนวนบิล,
            ROUND(SUM(f.line_revenue), 2)        AS ยอดขาย,
            ROUND(100.0 * SUM(f.line_revenue)
                  / SUM(SUM(f.line_revenue)) OVER (), 1) AS สัดส่วนร้อยละ
    FROM fact_sales_line f
    JOIN dim_time t ON f.time_key = t.time_key
    GROUP BY 1
    ORDER BY 3 DESC
""").fetchdf())

BQ3 — ช่วงเวลาใดหนาแน่นที่สุด



,ช่วงเวลา,จำนวนบิล,ยอดขาย,สัดส่วนร้อยละ
0,Dinner,8386,306378.60,37.5
1,Lunch,6206,262879.40,32.1
2,Afternoon,4860,182249.10,22.3
3,Late Night,1889,65966.30,8.1
4,Morning,9,386.65,0.0


In [49]:
print("BQ4 + BQ8 — ยอดขายรายเดือน เทียบยอดรวมกับยอดต่อวันทำการ")
print("(แสดงให้เห็นว่าทำไมต้องใช้ M6 แทนยอดรวมดิบ)\n")
display(con.execute("""
    SELECT  d.month_short                                   AS เดือน,
            COUNT(DISTINCT f.date_key)                      AS วันทำการ,
            ROUND(SUM(f.line_revenue), 2)                   AS ยอดขายรวม,
            ROUND(SUM(f.line_revenue)
                  / COUNT(DISTINCT f.date_key), 2)          AS ยอดขายต่อวันทำการ
    FROM fact_sales_line f
    JOIN dim_date d ON f.date_key = d.date_key
    GROUP BY d.month, d.month_short
    ORDER BY d.month
""").fetchdf())

BQ4 + BQ8 — ยอดขายรายเดือน เทียบยอดรวมกับยอดต่อวันทำการ
(แสดงให้เห็นว่าทำไมต้องใช้ M6 แทนยอดรวมดิบ)



,เดือน,วันทำการ,ยอดขายรวม,ยอดขายต่อวันทำการ
0,Jan,31,69793.30,2251.40
1,Feb,28,65159.60,2327.13
2,Mar,31,70397.10,2270.87
3,Apr,30,68736.80,2291.23
4,May,31,71402.75,2303.31
5,Jun,30,68230.20,2274.34
6,Jul,31,72557.90,2340.58
7,Aug,31,68278.25,2202.52
8,Sep,28,64180.05,2292.14
9,Oct,27,64027.60,2371.39


In [50]:
print("BQ6 — วันหยุดนักขัตฤกษ์ทำให้ยอดขายเปลี่ยนไปอย่างไร\n")
display(con.execute("""
    SELECT  CASE WHEN d.is_holiday THEN 'วันหยุดนักขัตฤกษ์'
                 WHEN d.is_weekend THEN 'สุดสัปดาห์'
                 ELSE 'วันธรรมดา' END                       AS ประเภทวัน,
            COUNT(DISTINCT f.date_key)                      AS จำนวนวัน,
            ROUND(SUM(f.line_revenue)
                  / COUNT(DISTINCT f.date_key), 2)          AS ยอดขายต่อวัน,
            ROUND(COUNT(DISTINCT f.order_id) * 1.0
                  / COUNT(DISTINCT f.date_key), 1)          AS บิลต่อวัน
    FROM fact_sales_line f
    JOIN dim_date d ON f.date_key = d.date_key
    GROUP BY 1
    ORDER BY 3 DESC
""").fetchdf())

BQ6 — วันหยุดนักขัตฤกษ์ทำให้ยอดขายเปลี่ยนไปอย่างไร



,ประเภทวัน,จำนวนวัน,ยอดขายต่อวัน,บิลต่อวัน
0,วันหยุดนักขัตฤกษ์,11,2636.55,67.0
1,วันธรรมดา,243,2331.16,61.0
2,สุดสัปดาห์,104,2138.33,55.6


In [51]:
print("โบนัส — วัตถุดิบที่ถูกใช้ในเมนูมากที่สุด (ใช้ Bridge Table)")
print("นับจำนวนเมนูที่ใช้ ไม่ใช่ยอดขาย เพื่อเลี่ยงการนับซ้ำ\n")
display(con.execute("""
    SELECT  i.ingredient_name       AS วัตถุดิบ,
            i.ingredient_category   AS หมวด,
            COUNT(*)                AS จำนวนเมนูที่ใช้
    FROM bridge_pizza_ingredient b
    JOIN dim_ingredient i ON b.ingredient_key = i.ingredient_key
    GROUP BY 1, 2
    ORDER BY 3 DESC
    LIMIT 10
""").fetchdf())

โบนัส — วัตถุดิบที่ถูกใช้ในเมนูมากที่สุด (ใช้ Bridge Table)
นับจำนวนเมนูที่ใช้ ไม่ใช่ยอดขาย เพื่อเลี่ยงการนับซ้ำ



,วัตถุดิบ,หมวด,จำนวนเมนูที่ใช้
0,Garlic,Vegetable & Other,20
1,Tomatoes,Vegetable & Other,18
2,Red Onions,Vegetable & Other,13
3,Red Peppers,Vegetable & Other,10
4,Spinach,Vegetable & Other,8
5,Mushrooms,Vegetable & Other,7
6,Mozzarella Cheese,Cheese,6
7,Pepperoni,Meat,6
8,Chicken,Meat,5
9,Artichokes,Vegetable & Other,5


## 6.2 สรุปผลการทำงานของ Pipeline

In [52]:
summary = con.execute("""
    SELECT 'fact_sales_line' AS ตาราง, COUNT(*) AS จำนวนแถว FROM fact_sales_line
    UNION ALL SELECT 'dim_date',                COUNT(*) FROM dim_date
    UNION ALL SELECT 'dim_time',                COUNT(*) FROM dim_time
    UNION ALL SELECT 'dim_pizza',               COUNT(*) FROM dim_pizza
    UNION ALL SELECT 'dim_weather',             COUNT(*) FROM dim_weather
    UNION ALL SELECT 'dim_ingredient',          COUNT(*) FROM dim_ingredient
    UNION ALL SELECT 'bridge_pizza_ingredient', COUNT(*) FROM bridge_pizza_ingredient
""").fetchdf()
display(summary)

print("\n--- บันทึกการแก้ไขปัญหาคุณภาพข้อมูล ---")
display(pd.DataFrame(AUDIT))

con.close()
print(f"\nปิดการเชื่อมต่อแล้ว | คลังข้อมูล: {DB_PATH.name} "
      f"({DB_PATH.stat().st_size/1024:.0f} KB)")
print("\nETL PIPELINE เสร็จสมบูรณ์")

,ตาราง,จำนวนแถว
0,fact_sales_line,48620
1,dim_date,365
2,dim_time,24
3,dim_pizza,96
4,dim_weather,23
5,dim_ingredient,65
6,bridge_pizza_ingredient,181



--- บันทึกการแก้ไขปัญหาคุณภาพข้อมูล ---


,ขั้นตอน,ประเภทปัญหา,รายละเอียด,ก่อน,หลัง,วิธีแก้
0,Extract,Encoding ไม่ใช่ UTF-8,pizza_types.csv มี byte 0x91 (smart quote แบบ ...,อ่านไม่ได้ (UnicodeDecodeError),อ่านได้ 32 แถว,ระบุ encoding='cp1252' ตอนอ่านไฟล์
1,Extract,โครงสร้าง Columnar ไม่ใช่ Row,Open-Meteo คืน array คู่ขนาน โหลดเข้า DW ตรงๆ ...,9 arrays คู่ขนาน,"8,760 แถว x 9 คอลัมน์",Transpose ด้วย pd.DataFrame() ให้เป็น row-orie...
2,Clean,ข้อมูลซ้ำ (Duplicate Records),วันหยุดซ้ำวันที่เดียวกันจากการแยกตามรัฐ,16 แถว,13 แถว,ยุบเหลือ 1 แถวต่อวัน โดยให้ global=True มาก่อน
3,Clean,หน่วยวัดไม่ตรงกัน,"อุณหภูมิ C, ฝน mm, หิมะ cm, ลม km/h",หน่วยเมตริก (4 หน่วย),"หน่วยอังกฤษ (F, inch, mph)",แปลงหน่วยทั้งหมด โดยหิมะใช้ตัวหาร 2.54 แยกจากฝ...
4,Clean,รูปแบบวันที่ไม่ตรงกัน,3 แหล่งใช้รูปแบบวันที่ต่างกัน 3 แบบ,string 3 รูปแบบ,datetime64 ทั้งหมด,ระบุ format string ชัดเจนใน pd.to_datetime() ท...
5,Clean,ค่าที่เป็นรหัส (Coded Values),weather_code เป็นตัวเลข WMO ที่ตีความเองไม่ได้,13 รหัสตัวเลข,4 กลุ่มสภาพอากาศที่อ่านได้,join กับตารางอ้างอิง WMO 4677
6,Clean,Timezone / DST,API ใช้ UTC offset คงที่ ไม่ขยับตาม DST,UTC-5 ตลอดปี,บันทึกเป็นข้อจำกัด ไม่ปรับแก้ข้อมูล,ยอมรับความคลาด 1 ชม. ช่วงฤดูหนาว และระบุไว้ในร...
7,Transform,ค่าหลายค่าในช่องเดียว (ผิด 1NF),คอลัมน์ ingredients เก็บวัตถุดิบหลายชนิดคั่นด้...,32 แถว (ช่องละหลายค่า),181 แถว (ช่องละหนึ่งค่า),แตกด้วย str.split() + explode() แล้วทำชื่อให้เ...



ปิดการเชื่อมต่อแล้ว | คลังข้อมูล: pizza_dw.duckdb (2060 KB)

ETL PIPELINE เสร็จสมบูรณ์


---
# สรุป

| Requirement | สิ่งที่ทำในไฟล์นี้ |
|-------------|---------------------|
| ข้อมูลแต่ละแหล่งถูกนำเข้าอย่างไร | ส่วนที่ 1 — CSV เชิงสัมพันธ์ 4 ตาราง และ Nested JSON จาก 2 REST API |
| ปัญหาคุณภาพถูกตรวจพบและแก้อย่างไร | ส่วนที่ 2 — แก้ 7 ประเภท ทุกข้อมีหลักฐานก่อน–หลังใน Audit Log |
| ข้อมูลหลายแหล่งถูกเชื่อมโยงอย่างไร | ส่วนที่ 4 — เชื่อมยอดขายกับสภาพอากาศด้วยคีย์ วันที่ + ชั่วโมง |
| สร้างหรือคำนวณ Measures อย่างไร | ส่วนที่ 4.1 — แยก Base Measure กับ Derived Measure ตามหลักการ |
| ตรวจสอบความถูกต้องก่อนและหลัง Load | ส่วนที่ 6 — Control Totals และการตรวจ Referential Integrity |
| กระบวนการรันซ้ำได้ | Full refresh ทุกครั้ง ไม่มีขั้นตอนที่ต้องแก้ด้วยมือ |

### ขั้นตอนถัดไป
1. จัดทำตารางต้นทุนวัตถุดิบ เพื่อปลดล็อก **M7 กำไรขั้นต้น**
2. สร้าง Dashboard เชื่อมกับ `pizza_dw.duckdb`
3. เพิ่ม Fact Table ที่สองเพื่อรับคะแนนพิเศษ